# Verificação local — motor WISMO (pós-venda)

Este notebook simula o **fluxo do cliente usando o chatbot** e verifica, contra a API real, se o motor de status de entrega responde corretamente em diferentes datas de consulta.

Contexto completo da lógica, contrato das rotas e limites está em [`docs/pos-venda-wismo.md`](../docs/pos-venda-wismo.md). Este notebook não repete essa explicação — ele **testa** o que está descrito lá.

## Sobre a entrada usada aqui

**Ainda não existe o agente que interpreta uma pergunta livre** ("cadê meu pedido?", "meu pedido não chegou") e extrai o código do pedido dela. Enquanto esse agente não existe, este notebook — e a interface, quando for construída — usa uma **entrada já estruturada**: o mesmo formato que o agente devolveria depois de interpretar a pergunta.

```python
EntradaChat(
    pedido="ORD-DEMO-001",             # código do pedido (hoje digitado; no futuro, extraído pelo agente)
    pergunta_original="Cadê meu pedido?",  # o que o cliente teria digitado — só para o log do chat
    data_referencia=None,              # ISO 8601, opcional — simula o "slider de data" da demo
)
```

Isso mantém o teste desacoplado da parte que ainda não foi decidida (como o texto livre será interpretado) e focado no que já está pronto: o motor determinístico e o contrato HTTP.

## Pré-requisitos

1. Servidor local rodando com `DATA_SOURCE=local` (usa as fixtures de [`src/data/deliveries.ts`](../src/data/deliveries.ts), sem precisar de Supabase):
   ```bash
   cp .env.example .env.local   # já vem com DATA_SOURCE=local
   corepack pnpm dev
   ```
2. Pacotes Python: `requests`, `pandas`. Ambos já disponíveis neste ambiente.


In [1]:
import json
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from typing import Optional

import pandas as pd
import requests

BASE_URL = "http://localhost:3000"
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", None)


In [2]:
# Confere se o servidor está de pé antes de qualquer teste, com uma mensagem
# clara em vez de um traceback de conexão recusada.
try:
    resposta = requests.get(f"{BASE_URL}/api/wismo/orders/ORD-DEMO-001", timeout=3)
    resposta.raise_for_status()
    print("✅ Servidor respondendo em", BASE_URL)
except Exception as erro:
    raise RuntimeError(
        "Não consegui falar com o servidor local. Rode, num terminal separado:\n"
        "  cp .env.example .env.local   # se ainda não existir\n"
        "  corepack pnpm dev\n"
        f"Erro original: {erro}"
    )


✅ Servidor respondendo em http://localhost:3000


## Entrada padronizada e função de consulta

`EntradaChat` é o contrato de entrada descrito na introdução. `perguntar_ao_bot` chama a rota real (`GET /api/wismo/orders/:pedido`, com `asOf` opcional) e devolve a resposta crua — sem reinterpretar nada, para o teste ficar fiel ao que a interface receberia.


In [3]:
@dataclass
class EntradaChat:
    pedido: str
    pergunta_original: str
    data_referencia: Optional[str] = None  # ISO 8601; None = avaliar "agora"


def perguntar_ao_bot(entrada: EntradaChat) -> dict:
    params = {"asOf": entrada.data_referencia} if entrada.data_referencia else {}
    resposta = requests.get(f"{BASE_URL}/api/wismo/orders/{entrada.pedido}", params=params, timeout=10)
    try:
        corpo = resposta.json()
    except ValueError:
        corpo = {"error": resposta.text}
    return {"entrada": entrada, "http_status": resposta.status_code, "corpo": corpo}


def linha_estruturada(resultado: dict) -> dict:
    """Achata uma resposta em uma linha de tabela, estruturada e comparável."""
    entrada, http, corpo = resultado["entrada"], resultado["http_status"], resultado["corpo"]
    base = {
        "pedido": entrada.pedido,
        "data_referencia": entrada.data_referencia or "(agora)",
        "http": http,
    }
    if http != 200:
        return {**base, "erro": corpo.get("error"), "canal": None, "escada": None, "regua": None,
                "p75_prazo_dias": None, "fase": None, "flag": None, "atraso_dias": None,
                "escalar": None, "violacao_critica": None, "mensagem": None}
    status = corpo["status"]
    return {
        **base,
        "erro": None,
        "canal": corpo["order"]["channel"],
        "escada": ">".join(status["evidence"]["fallbackChain"]),
        "regua": f'{status["ruler"]["scope"]}:{status["ruler"]["scopeKey"]}',
        "p75_prazo_dias": status["ruler"]["p75Days"],
        "fase": status["phase"],
        "flag": status["flag"],
        "atraso_dias": status["daysLate"],
        "escalar": status["escalate"],
        "violacao_critica": status["criticalBreach"],
        "mensagem": status["message"],
    }


def imprimir_chat(resultado: dict) -> None:
    """Imprime a interação como apareceria na tela do cliente."""
    entrada, http, corpo = resultado["entrada"], resultado["http_status"], resultado["corpo"]
    print("─" * 72)
    if entrada.data_referencia:
        print(f"🕒 [modo demonstração — reproduzindo o pedido em {entrada.data_referencia}]")
    print(f"👤 Cliente: {entrada.pergunta_original}")
    mensagem = corpo["status"]["message"] if http == 200 else corpo.get("error", "(sem resposta)")
    print(f"🤖 MIPO: {mensagem}")
    print("─" * 72)


## Cenário 1 — quatro pedidos reais, sem controle de data

Simula quatro clientes diferentes perguntando "cadê meu pedido?" hoje, cada um com um código de pedido distinto. Nenhum usa `data_referencia` — a avaliação é feita na data atual, e como todos são pedidos históricos (2023), todos devem sair como `entregue`.


In [4]:
pedidos_demo = ["ORD-DEMO-001", "ORD-DEMO-002", "ORD-DEMO-003", "ORD-DEMO-004"]

entradas_cenario_1 = [
    EntradaChat(pedido=codigo, pergunta_original=f"Cadê meu pedido? {codigo}")
    for codigo in pedidos_demo
]

resultados_cenario_1 = [perguntar_ao_bot(entrada) for entrada in entradas_cenario_1]
tabela_1 = pd.DataFrame([linha_estruturada(resultado) for resultado in resultados_cenario_1])
tabela_1


,pedido,data_referencia,http,erro,canal,escada,regua,p75_prazo_dias,fase,flag,atraso_dias,escalar,violacao_critica,mensagem
0,ORD-DEMO-001,(agora),200,None,Orgânico,channel,channel:Orgânico,11,delivered,on_time,0,False,False,"Seu pedido foi entregue em 6 dias, dentro do prazo de 11..."
1,ORD-DEMO-002,(agora),200,None,Orgânico,channel,channel:Orgânico,11,delivered,late,3,False,True,"Seu pedido foi entregue em 14 dias, 3 dias além do prazo..."
2,ORD-DEMO-003,(agora),200,None,Marketplace,channel,channel:Marketplace,15,delivered,late,5,False,True,"Seu pedido foi entregue em 20 dias, 5 dias além do prazo..."
3,ORD-DEMO-004,(agora),200,None,NaN,channel>state,state:AC,12,delivered,on_time,0,False,False,"Seu pedido foi entregue em 9 dias, dentro do prazo de 12..."


Leitura esperada:

- **ORD-DEMO-001** (Orgânico): entregue em 6 dias, prazo do canal é 11 → `on_time`.
- **ORD-DEMO-002** (Orgânico): entregue em 14 dias, mesmo prazo de 11 → `late`, 3 dias de atraso, e como 14 já passa do limiar crítico do canal (13), `violacao_critica = True`.
- **ORD-DEMO-003** (Marketplace): entregue em 20 dias, prazo do canal é 15 → `late`, 5 dias de atraso, também crítico.
- **ORD-DEMO-004** (sem canal registrado): a escada pula o canal e cai para o estado (`escada = channel>state`) — mostra a rede de segurança funcionando.


## Cenário 2 — o mesmo pedido, percorrido no tempo

Em vez de fixar os deslocamentos de dia, busco primeiro a data real do pedido (`orderedAt`) e construo `data_referencia` a partir dela. Isso evita hardcode e usa a própria API como fonte da verdade.

Esse é o mecanismo por trás do "slider de data" sugerido no doc para a demo: o mesmo pedido real, observado em pontos diferentes da própria jornada.


In [6]:
pedido_alvo = "ORD-DEMO-003"

# Busca o pedido sem asOf só para descobrir orderedAt e o resultado final real.
resultado_base = perguntar_ao_bot(EntradaChat(pedido=pedido_alvo, pergunta_original="(consulta de referência)"))
ordered_at = datetime.fromisoformat(resultado_base["corpo"]["order"]["orderedAt"].replace("Z", "+00:00"))
entrega_real_dias = resultado_base["corpo"]["order"]["actualDeliveryDays"]
print(f"Pedido {pedido_alvo}: feito em {ordered_at.isoformat()}, entregue em {entrega_real_dias} dias (fato real, não projetado).")


Pedido ORD-DEMO-003: feito em 2023-07-02T00:00:00+00:00, entregue em 20 dias (fato real, não projetado).


In [7]:
deslocamentos_dias = [0, 3, 16, 18, 20, 21]

entradas_cenario_2 = [
    EntradaChat(
        pedido=pedido_alvo,
        pergunta_original="Cadê meu pedido?",
        data_referencia=(ordered_at + timedelta(days=deslocamento)).isoformat().replace("+00:00", "Z"),
    )
    for deslocamento in deslocamentos_dias
]

resultados_cenario_2 = [perguntar_ao_bot(entrada) for entrada in entradas_cenario_2]
tabela_2 = pd.DataFrame([linha_estruturada(resultado) for resultado in resultados_cenario_2])
tabela_2.insert(1, "dias_desde_pedido", deslocamentos_dias)
tabela_2


,pedido,dias_desde_pedido,data_referencia,http,erro,canal,escada,regua,p75_prazo_dias,fase,flag,atraso_dias,escalar,violacao_critica,mensagem
0,ORD-DEMO-003,0,2023-07-02T00:00:00Z,200,None,Marketplace,channel,channel:Marketplace,15,preparing,on_time,0,False,False,Seu pedido está em preparação. A previsão de entrega é 1...
1,ORD-DEMO-003,3,2023-07-05T00:00:00Z,200,None,Marketplace,channel,channel:Marketplace,15,in_transit,on_time,0,False,False,Seu pedido está em transporte. A previsão de entrega é 1...
2,ORD-DEMO-003,16,2023-07-18T00:00:00Z,200,None,Marketplace,channel,channel:Marketplace,15,in_transit,late,1,False,False,Seu pedido está em transporte e já passou 1 dia do prazo...
3,ORD-DEMO-003,18,2023-07-20T00:00:00Z,200,None,Marketplace,channel,channel:Marketplace,15,in_transit,late,3,True,True,Seu pedido está em transporte e já passou 3 dias do praz...
4,ORD-DEMO-003,20,2023-07-22T00:00:00Z,200,None,Marketplace,channel,channel:Marketplace,15,delivered,late,5,False,True,"Seu pedido foi entregue em 20 dias, 5 dias além do prazo..."
5,ORD-DEMO-003,21,2023-07-23T00:00:00Z,200,None,Marketplace,channel,channel:Marketplace,15,delivered,late,5,False,True,"Seu pedido foi entregue em 20 dias, 5 dias além do prazo..."


A evolução esperada, para este pedido (canal Marketplace, prazo do canal = 15 dias, limiar crítico = 17):

| dia | fase | flag | atraso | escalar |
|---:|---|---|---:|---|
| 0 | `preparing` | `on_time` | 0 | não |
| 3 | `in_transit` | `on_time` | 0 | não |
| 16 | `in_transit` | `late` | 1 | não |
| 18 | `in_transit` | `late` | 3 | **sim** |
| 20 | `delivered` | `late` | 5 | não (já entregue) |
| 21 | `delivered` | `late` | 5 | não |

No dia 20 o pedido cruza a data real de entrega e a resposta passa a refletir o fato consumado — o veredito de `atraso` fica congelado em 5 dias para qualquer data de referência a partir daí, porque a entrega já aconteceu e não depende mais de "agora".


### Um ponto para quem for construir o frontend

A coluna abaixo mostra a diferença entre o campo bruto `order.actualDeliveryDays` (que **não é projetado** pela data de referência) e o campo `elapsedDays` de dentro da evidência (que **é** projetado). Isso é a pendência já registrada em [`docs/pos-venda-wismo.md`](../docs/pos-venda-wismo.md), seção 7.3, item 4 — reproduzida aqui de forma concreta.


In [7]:
comparacao = []
for resultado in resultados_cenario_2:
    corpo = resultado["corpo"]
    comparacao.append({
        "data_referencia": resultado["entrada"].data_referencia,
        "actualDeliveryDays_bruto (NÃO projetado)": corpo["order"]["actualDeliveryDays"],
        "evidence.elapsedDays (projetado)": corpo["status"]["evidence"]["elapsedDays"],
        "fase_resultante": corpo["status"]["phase"],
    })
pd.DataFrame(comparacao)


        data_referencia  actualDeliveryDays_bruto (NÃO projetado)  evidence.elapsedDays (projetado) fase_resultante
0  2023-07-02T00:00:00Z                                        20                                 0       preparing
1  2023-07-05T00:00:00Z                                        20                                 3      in_transit
2  2023-07-18T00:00:00Z                                        20                                16      in_transit
3  2023-07-20T00:00:00Z                                        20                                18      in_transit
4  2023-07-22T00:00:00Z                                        20                                20       delivered
5  2023-07-23T00:00:00Z                                        20                                20       delivered

Note que `actualDeliveryDays_bruto` é sempre 20, em qualquer linha — mesmo quando a fase resultante é `preparing` ou `in_transit`. Se a interface exibir esse campo diretamente enquanto `asOf` está ativo, ela revela ao usuário quantos dias o pedido *realmente* levou, o que não deveria ser visível num cenário que está sendo reproduzido como "ainda em andamento". **A interface deve usar `elapsedDays`, nunca `actualDeliveryDays`, quando `data_referencia` estiver em uso.**


## Cenário 3 — simulação da conversa no chat

Para dar a sensação real do fluxo do cliente, abaixo estão três momentos impressos como a tela do chat mostraria: uma consulta sem histórico de atraso, uma com violação crítica, e o mesmo pedido do Cenário 2 reproduzido em pleno atraso.


In [8]:
conversa = [
    EntradaChat(pedido="ORD-DEMO-001", pergunta_original="Oi, cadê o meu pedido?"),
    EntradaChat(pedido="ORD-DEMO-002", pergunta_original="Meu pedido ainda não chegou, o que aconteceu?"),
    EntradaChat(
        pedido="ORD-DEMO-003",
        pergunta_original="E aí, alguma notícia do meu pedido?",
        data_referencia=(ordered_at + timedelta(days=18)).isoformat().replace("+00:00", "Z"),
    ),
]

for entrada in conversa:
    imprimir_chat(perguntar_ao_bot(entrada))


────────────────────────────────────────────────────────────────────────
👤 Cliente: Oi, cadê o meu pedido?
🤖 MIPO: Seu pedido foi entregue em 6 dias, dentro do prazo de 11 dias estimado para o canal Orgânico a partir do histórico.
────────────────────────────────────────────────────────────────────────
────────────────────────────────────────────────────────────────────────
👤 Cliente: Meu pedido ainda não chegou, o que aconteceu?
🤖 MIPO: Seu pedido foi entregue em 14 dias, 3 dias além do prazo de 11 dias estimado para o canal Orgânico. Isso ficou acima do limiar de 13 dias usado para acionar o time responsável.
────────────────────────────────────────────────────────────────────────
────────────────────────────────────────────────────────────────────────
🕒 [modo demonstração — reproduzindo o pedido em 2023-07-20T00:00:00Z]
👤 Cliente: E aí, alguma notícia do meu pedido?
🤖 MIPO: Seu pedido está em transporte e já passou 3 dias do prazo previsto (17/07). Vamos acionar o time responsável.


## Cenário 4 — casos de erro

Pedido inexistente e data de referência inválida (anterior à data do pedido) precisam falhar de forma previsível, com mensagem em pt-BR e o código HTTP correto — é o que a interface vai usar para decidir qual tela de erro mostrar.


In [11]:
entradas_erro = [
    EntradaChat(pedido="ORD-FANTASMA", pergunta_original="Cadê meu pedido? ORD-FANTASMA"),
    EntradaChat(
        pedido="ORD-DEMO-003",
        pergunta_original="(data de referência anterior ao pedido, deve ser rejeitada)",
        data_referencia="2020-01-01T00:00:00Z",
    ),
]

resultados_erro = [perguntar_ao_bot(entrada) for entrada in entradas_erro]
pd.DataFrame([linha_estruturada(resultado) for resultado in resultados_erro])[["pedido", "data_referencia", "http", "erro"]]


,pedido,data_referencia,http,erro
0,ORD-FANTASMA,(agora),404,Pedido não encontrado.
1,ORD-DEMO-003,2020-01-01T00:00:00Z,400,Data de referência inválida para este pedido.


## Checklist de verificação automática

Os valores abaixo foram confirmados manualmente contra a API antes de este notebook ser escrito. Rodar esta célula é uma verificação de regressão: se algo no motor, na régua local ou no contrato HTTP mudar sem querer, uma linha aqui vira ❌.


In [12]:
esperado = [
    {"pedido": "ORD-DEMO-001", "escada": "channel", "fase": "delivered", "flag": "on_time", "atraso_dias": 0, "violacao_critica": False},
    {"pedido": "ORD-DEMO-002", "escada": "channel", "fase": "delivered", "flag": "late", "atraso_dias": 3, "violacao_critica": True},
    {"pedido": "ORD-DEMO-003", "escada": "channel", "fase": "delivered", "flag": "late", "atraso_dias": 5, "violacao_critica": True},
    {"pedido": "ORD-DEMO-004", "escada": "channel>state", "fase": "delivered", "flag": "on_time", "atraso_dias": 0, "violacao_critica": False},
]

obtido_por_pedido = {linha["pedido"]: linha for linha in tabela_1.to_dict("records")}

tudo_ok = True
for esperado_linha in esperado:
    obtido = obtido_por_pedido[esperado_linha["pedido"]]
    campos_ok = all(obtido[campo] == esperado_linha[campo] for campo in esperado_linha if campo != "pedido")
    simbolo = "✅" if campos_ok else "❌"
    tudo_ok &= campos_ok
    print(f"{simbolo} {esperado_linha['pedido']}: esperado={ {k: v for k, v in esperado_linha.items() if k != 'pedido'} }")
    if not campos_ok:
        print(f"    obtido={ {k: obtido[k] for k in esperado_linha if k != 'pedido'} }")

print()
print("✅ Cenário 1 confere com o esperado." if tudo_ok else "❌ Divergência encontrada — ver acima.")


✅ ORD-DEMO-001: esperado={'escada': 'channel', 'fase': 'delivered', 'flag': 'on_time', 'atraso_dias': 0, 'violacao_critica': False}
✅ ORD-DEMO-002: esperado={'escada': 'channel', 'fase': 'delivered', 'flag': 'late', 'atraso_dias': 3, 'violacao_critica': True}
✅ ORD-DEMO-003: esperado={'escada': 'channel', 'fase': 'delivered', 'flag': 'late', 'atraso_dias': 5, 'violacao_critica': True}
✅ ORD-DEMO-004: esperado={'escada': 'channel>state', 'fase': 'delivered', 'flag': 'on_time', 'atraso_dias': 0, 'violacao_critica': False}

✅ Cenário 1 confere com o esperado.


In [13]:
esperado_evolucao = {
    0: {"fase": "preparing", "flag": "on_time", "atraso_dias": 0, "escalar": False},
    3: {"fase": "in_transit", "flag": "on_time", "atraso_dias": 0, "escalar": False},
    16: {"fase": "in_transit", "flag": "late", "atraso_dias": 1, "escalar": False},
    18: {"fase": "in_transit", "flag": "late", "atraso_dias": 3, "escalar": True},
    20: {"fase": "delivered", "flag": "late", "atraso_dias": 5, "escalar": False},
    21: {"fase": "delivered", "flag": "late", "atraso_dias": 5, "escalar": False},
}

obtido_por_dia = dict(zip(deslocamentos_dias, tabela_2.to_dict("records")))

tudo_ok = True
for dia, esperado_linha in esperado_evolucao.items():
    obtido = obtido_por_dia[dia]
    campos_ok = all(obtido[campo] == valor for campo, valor in esperado_linha.items())
    simbolo = "✅" if campos_ok else "❌"
    tudo_ok &= campos_ok
    print(f"{simbolo} dia {dia}: esperado={esperado_linha}")
    if not campos_ok:
        print(f"    obtido={ {k: obtido[k] for k in esperado_linha} }")

print()
print("✅ Cenário 2 (evolução no tempo) confere com o esperado." if tudo_ok else "❌ Divergência encontrada — ver acima.")


✅ dia 0: esperado={'fase': 'preparing', 'flag': 'on_time', 'atraso_dias': 0, 'escalar': False}
✅ dia 3: esperado={'fase': 'in_transit', 'flag': 'on_time', 'atraso_dias': 0, 'escalar': False}
✅ dia 16: esperado={'fase': 'in_transit', 'flag': 'late', 'atraso_dias': 1, 'escalar': False}
✅ dia 18: esperado={'fase': 'in_transit', 'flag': 'late', 'atraso_dias': 3, 'escalar': True}
✅ dia 20: esperado={'fase': 'delivered', 'flag': 'late', 'atraso_dias': 5, 'escalar': False}
✅ dia 21: esperado={'fase': 'delivered', 'flag': 'late', 'atraso_dias': 5, 'escalar': False}

✅ Cenário 2 (evolução no tempo) confere com o esperado.


In [14]:
codigos_erro_esperados = {"ORD-FANTASMA": 404, "ORD-DEMO-003": 400}
tudo_ok = True
for resultado in resultados_erro:
    pedido = resultado["entrada"].pedido
    esperado_http = codigos_erro_esperados[pedido]
    ok = resultado["http_status"] == esperado_http
    tudo_ok &= ok
    print(f"{'✅' if ok else '❌'} {pedido}: esperado HTTP {esperado_http}, obtido {resultado['http_status']}")

print()
print("✅ Cenário 4 (erros) confere com o esperado." if tudo_ok else "❌ Divergência encontrada — ver acima.")


✅ ORD-FANTASMA: esperado HTTP 404, obtido 404
✅ ORD-DEMO-003: esperado HTTP 400, obtido 400

✅ Cenário 4 (erros) confere com o esperado.


## Conclusão

Se todas as células de verificação acima mostraram ✅, o motor determinístico, a escada de fallback, o cálculo de atraso/escalonamento e o recurso de data de referência estão funcionando de acordo com o que está documentado em [`docs/pos-venda-wismo.md`](../docs/pos-venda-wismo.md).

**O que este notebook não testa:** interface de chat (não existe ainda), agente de interpretação de texto livre (não existe ainda) e o modo `DATA_SOURCE=supabase` (precisa de credenciais reais e da régua importada dos CSVs do case). Para testar contra o Supabase, troque `DATA_SOURCE=local` por `DATA_SOURCE=supabase` no `.env.local`, rode as migrations e a importação, e repita este notebook — o contrato das rotas é idêntico nos dois modos.
